# Project onto the span of dependent vectors

Projecting a vector onto the span of a few others is least squares. The formula
$A(A^\top A)^{-1}A^\top$ breaks when the vectors are dependent, because $A^\top A$ then
has no inverse, and that happens in practice: two features that always move together, or a
direction that was added twice. Here you build a basis that drops the redundant vectors, by
Gram-Schmidt, and project with it.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Four sensor channels recorded at 5 times, as columns. Channel 2 is channel 0 plus
# channel 1, and channel 3 is channel 0 recorded again at twice the gain.
V = np.array([[1.0, 0.0, 1.0, 2.0],
              [1.0, 1.0, 2.0, 2.0],
              [0.0, 1.0, 1.0, 0.0],
              [2.0, 0.0, 2.0, 4.0],
              [1.0, 3.0, 4.0, 2.0]])
x = np.array([3.0, 2.0, 1.0, 1.0, 4.0])
print("V", V.shape, " rank", np.linalg.matrix_rank(V))
print("det(V.T @ V) =", round(abs(float(np.linalg.det(V.T @ V))), 6))

## Part 1 (coding task): An orthonormal basis that skips redundant columns

Write `orthonormal_basis(V, tol=1e-10)`. `V` is $n \times k$. Return `Q` of shape
$n \times r$ whose columns are perpendicular unit vectors spanning the same space as the
columns of `V`, where $r$ is the rank.

Use **modified** Gram-Schmidt, taking the columns of `V` in order. For column $j$, start
with $\mathbf{w}$ equal to it, and for each basis vector $\mathbf{q}$ found so far subtract
$(\mathbf{q} \cdot \mathbf{w})\,\mathbf{q}$, using the current $\mathbf{w}$ each time. If
what is left has length at most `tol` times the length of the original column (or the
column is zero), the column adds no new direction: skip it. Otherwise divide $\mathbf{w}$
by its length and add it to the basis. With no columns kept, return shape $(n, 0)$.

In [ ]:
def orthonormal_basis(V, tol=1e-10):
    V = np.asarray(V, dtype=float)
    basis = []
    # TODO: modified Gram-Schmidt over the columns of V, skipping the ones that add
    #       no new direction.
    raise NotImplementedError

In [ ]:
Q = orthonormal_basis(V)
assert not np.isnan(Q).any(), "Q has nan: a dependent column left (almost) nothing, and it was divided by that length"
assert Q.shape == (5, 2), f"V has rank 2, so Q should be 5 x 2, got {Q.shape}"
assert np.allclose(Q.T @ Q, np.eye(2)), "the columns of Q are not perpendicular unit vectors"
assert np.allclose(Q[:, 0], V[:, 0] / np.linalg.norm(V[:, 0])), "the first basis vector should be the first column of V, scaled to length 1"
assert np.allclose(Q @ (Q.T @ V), V), "some column of V is not in the span of Q"
Z = orthonormal_basis(np.array([[0.0, 2.0, 0.0], [0.0, 0.0, 0.0], [0.0, 1.0, 0.0]]))
assert Z.shape == (3, 1), f"zero columns add no direction: expected shape (3, 1), got {Z.shape}"
assert orthonormal_basis(np.zeros((4, 2))).shape == (4, 0), "with no direction at all, return shape (n, 0)"
P = np.vander(np.linspace(0, 1, 50), 10, increasing=True)   # 1, t, t^2, ..., t^9: nearly dependent
QP = orthonormal_basis(P)
assert QP.shape == (50, 10), f"the 10 powers of t are independent, so all 10 should be kept, got {QP.shape[1]}"
err = np.linalg.norm(QP.T @ QP - np.eye(10))
assert err < 1e-6, f"on the powers of t, Q.T @ Q is off from I by {err:.1e}: subtract the projections from what is left of the column (modified Gram-Schmidt), not from the original column"
print("Part 1 passed")

## Part 2 (coding task): Project onto the span

Write `project(V, x)`, which returns the point in the span of the columns of `V` closest
to `x`. Use `orthonormal_basis`: with perpendicular unit vectors the projection is the sum
of the projections onto each of them. The columns of `V` may be dependent, so do not form
$(V^\top V)^{-1}$.

In [ ]:
def project(V, x):
    # TODO: project x onto the span of the columns of V.
    raise NotImplementedError

In [ ]:
p = np.asarray(project(V, x), dtype=float)
assert p.shape == (5,), f"the projection is a vector like x, shape (5,), got {p.shape}"
r = x - p
assert np.allclose(V.T @ r, 0), f"the residual x - p is not perpendicular to every column of V: V.T @ r = {np.round(V.T @ r, 4)}"
coef, *_ = np.linalg.lstsq(V, p, rcond=None)
assert np.allclose(V @ coef, p), "p is not in the span of the columns of V"
assert np.allclose(p, [1.0, 2.0, 1.0, 2.0, 4.0]), f"got {np.round(p, 4)}"
assert np.allclose(project(V, V[:, 2]), V[:, 2]), "a vector already in the span should come back unchanged"
assert np.allclose(project(V[:, :1], np.array([1.0, -1.0, 0.0, 0.0, 0.0])), 0), "a vector perpendicular to the span projects to zero"
print("projection:", np.round(p, 4), "  residual length:", round(float(np.linalg.norm(r)), 4))
print("Part 2 passed")

## Solution

### Part 1: An orthonormal basis that skips redundant columns

In [ ]:
def orthonormal_basis(V, tol=1e-10):
    V = np.asarray(V, dtype=float)
    basis = []
    for j in range(V.shape[1]):
        column = V[:, j]
        w = column.copy()
        for q in basis:
            w -= (q @ w) * q              # modified: project what is left, not the column
        length = np.linalg.norm(w)
        if length == 0 or length <= tol * np.linalg.norm(column):
            continue                      # nothing new: a combination of earlier columns
        basis.append(w / length)
    if not basis:
        return np.zeros((V.shape[0], 0))
    return np.column_stack(basis)

**Why it works.** 
Each subtraction removes the part of $\mathbf{w}$ along one basis vector, so what is
left is perpendicular to all of them, the residual from projecting the column onto the span
so far. A column that is a combination of earlier ones leaves a residual of zero, or of
rounding size, and is skipped, so the basis has exactly rank-many vectors. Comparing with
`tol` times the column's own length makes the test independent of units. Computing each
coefficient from the current $\mathbf{w}$ also removes any part along $\mathbf{q}$ that
rounding put back during earlier steps. The classical version, which uses the original
column, is off from orthogonal by about $10^{-2}$ on the powers of $t$, and this one by
about $10^{-10}$.

### Part 2: Project onto the span

In [ ]:
def project(V, x):
    Q = orthonormal_basis(V)
    return Q @ (Q.T @ np.asarray(x, dtype=float))

**Why it works.** 
For perpendicular unit vectors $\mathbf{q}_1, \dots, \mathbf{q}_r$ the normal equations
have $Q^\top Q = I$, so the coefficients are just $Q^\top \mathbf{x}$ and the projection is
$QQ^\top\mathbf{x}$. The basis spans the same space as the columns of `V`, so this is also
the projection onto their span, and the redundant channels change nothing. Adding up the
projections onto the columns of `V` themselves would be wrong here: they are not
perpendicular, so the sum counts shared directions more than once and its residual is not
perpendicular to the columns.